# Transfer learning & fine-tuning

_Few-shot & Transfer Learning_

**Borrow a model that already learned to see, then teach it your tiny new task with almost no data.**

Transfer learning means you take a model that was already trained on a huge dataset, and reuse what it learned for a brand-new task that has very little data.

       The big model already knows useful things: edges, textures, shapes, word patterns. You do not throw that away. You keep it and bolt on a small new piece for your task.

---

This notebook is a practice scaffold for the **Transfer learning & fine-tuning** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Visualize the data & results

_On real digit images, does reusing a frozen pretrained representation beat training from scratch when you only have a few labels per class?_

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

digits = load_digits()                      # 1797 real 8x8 handwritten digits
X = digits.data / 16.0                       # pixels scaled to 0..1
y = digits.target
tgt = np.isin(y, [5, 6, 7, 8, 9])            # the NEW task: classify digits 5-9

# hold out a fixed test set the few-shot learner never sees
Xrest, Xte, yrest, yte = train_test_split(X[tgt], y[tgt], test_size=0.35,
                                          random_state=0, stratify=y[tgt])
# split the rest: an ABUNDANT source pool (pretraining) + a disjoint few-shot label pool
Xsrc, Xpool, ysrc, ypool = train_test_split(Xrest, yrest, test_size=0.5,
                                            random_state=1, stratify=yrest)

# PRETRAIN a backbone on the abundant source labels, then FREEZE its hidden layer
backbone = MLPClassifier(hidden_layer_sizes=(64,), max_iter=1000,
                         random_state=0, alpha=1e-3).fit(Xsrc, ysrc)

def features(A):                             # forward pass through frozen hidden layers
    a = A
    for W, b in zip(backbone.coefs_[:-1], backbone.intercepts_[:-1]):
        a = np.maximum(0, a @ W + b)         # ReLU activations = the representation
    return a

F_pool, F_te = features(Xpool), features(Xte)

def pick(k, seed):                           # k labeled examples per class
    rs = np.random.default_rng(seed)
    idx = []
    for c in np.unique(ypool):
        ci = np.where(ypool == c)[0]
        idx.extend(rs.choice(ci, size=min(k, len(ci)), replace=False))
    return np.array(idx)

shots = [1, 2, 5, 10, 20]
transfer, scratch = [], []
for k in shots:
    ta, sa = [], []
    for seed in range(40):                   # average over many label draws
        idx = pick(k, seed); yk = ypool[idx]
        if len(np.unique(yk)) < 2:
            continue
        # transfer: linear head on FROZEN pretrained features
        ta.append(LogisticRegression(max_iter=3000).fit(F_pool[idx], yk).score(F_te, yte))
        # from scratch: same head, but on raw pixels (no pretrained representation)
        sa.append(LogisticRegression(max_iter=3000).fit(Xpool[idx], yk).score(Xte, yte))
    transfer.append(round(float(np.mean(ta)), 3))
    scratch.append(round(float(np.mean(sa)), 3))

print("shots   ", shots)
print("transfer", transfer)   # [0.907, 0.945, 0.962, 0.972, 0.977]
print("scratch ", scratch)    # [0.703, 0.807, 0.895, 0.936, 0.951]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have a model pretrained on millions of photos and a new task with only 40 labeled images. Should you do feature extraction (freeze the backbone) or full fine-tuning (train every layer)? Why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count your data versus the number of weights. — _40 images is tiny. The backbone has millions of weights. Training all of them on 40 images would overfit badly._
- Choose feature extraction: freeze the backbone, train only a small new head. — _The frozen backbone already gives good features. The head has few weights, so 40 images can fit it._
- Optionally fine-tune the top one or two layers later, with a very small learning rate. — _Once the head is stable, gentle fine-tuning can squeeze out a little more, without forgetting the general features._

**Answer:** Feature extraction. With only 40 images, freeze the backbone and train a small head; full fine-tuning would overfit the millions of backbone weights.

</details>

**Problem 2.** While fine-tuning, an engineer uses a large learning rate to "speed things up". Accuracy on the original general benchmark crashes. What happened, and what is the fix?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Name the symptom: the model lost skills it used to have. — _This is catastrophic forgetting: big steps overwrote the pretrained weights that held the general knowledge._
- Identify the cause: the learning rate was too large for fine-tuning. — _The backbone weights were already in a good spot. A large step $\eta$ shoves them far away and destroys that._
- Fix it: use a much smaller learning rate (and optionally freeze the lower layers). — _A small $\eta$ nudges the weights only slightly, so the old general features survive while the new task is learned._

**Answer:** Catastrophic forgetting from too-large a learning rate. Fix: fine-tune with a small learning rate (and freeze lower layers) so the pretrained features are preserved.

</details>